# 01 — Ingestão e limpeza das escolas
**Universo:** rede municipal completa de Educação Infantil e Ensino Fundamental — **direta + conveniada/parceira**
(privadas ficam fora).

Fonte principal: *Cadastro de escolas municipais, conveniadas e privadas* (SME, Portal de Dados Abertos),
que já traz coordenadas, tipo de escola, rede e código INEP.
Fonte de conferência: camadas do GeoSampa (Educação Infantil / Ensino Fundamental e Médio) em `data/raw/geosampa/*.gpkg`.

Saída: `data/interim/escolas.gpkg` (camada `escolas` + camadas `buffer_50m … buffer_250m`)

In [ ]:
!pip -q install geopandas pyogrio mapclassify

In [ ]:
# --- setup comum (Colab) ---
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/2_PESQUISA/02_PROJETOS-PESQUISA-MODELAGEM/ESCOLAS-CONFORTO-TERMICO-SME'
import os, sys
os.chdir(ROOT); sys.path.append(f'{ROOT}/src')
RAW, INT, PRO, OUT = [f'{ROOT}/{p}' for p in ['data/raw', 'data/interim', 'data/processed', 'outputs']]
for p in [RAW, INT, PRO, OUT]:
    os.makedirs(p, exist_ok=True)
CRS = 'EPSG:31983'          # SIRGAS 2000 / UTM 23S
BUFFERS = [50, 100, 150, 250]
BUFFER_REF = 100            # buffer usado no score (≈ 1 pixel termal nativo do Landsat)

In [ ]:
import requests, glob, re, io
import pandas as pd, geopandas as gpd, numpy as np

API = 'https://dados.prefeitura.sp.gov.br/api/3/action/package_show'
pkg = requests.get(API, params={'id': 'cadastro-de-escolas-municipais-conveniadas-e-privadas'}, timeout=60).json()['result']
recursos = sorted([r for r in pkg['resources'] if 'csv' in (r.get('format') or '').lower() or r['url'].lower().endswith('.csv')],
                  key=lambda r: r.get('created') or '', reverse=True)
for r in recursos[:8]:
    print(r.get('created', '')[:10], '|', r['name'], '|', r['url'])

In [ ]:
# baixa o recurso mais recente (troque o índice se quiser outra data-base)
rec = recursos[0]
os.makedirs(f'{RAW}/sme', exist_ok=True)
dest = f"{RAW}/sme/cadastro_escolas_{(rec.get('created') or '')[:10]}.csv"
open(dest, 'wb').write(requests.get(rec['url'], timeout=300).content)

def ler_csv(p):
    for enc in ['utf-8', 'latin-1']:
        for sep in [';', ',']:
            try:
                df = pd.read_csv(p, sep=sep, encoding=enc, dtype=str)
                if df.shape[1] > 5:
                    return df
            except Exception:
                pass
    raise ValueError('não consegui ler ' + p)

cad = ler_csv(dest)
cad.columns = [c.strip().lower() for c in cad.columns]
print(cad.shape); print(list(cad.columns))

In [ ]:
# inspeção das categorias — CONFIRA antes de filtrar
for c in ['rede', 'tipoesc', 'situacao']:
    if c in cad.columns:
        print(f'\n== {c} ==')
        print(cad[c].value_counts(dropna=False).head(40))

### Regras de filtro
- `situacao` ativa
- `rede` ≠ privada (mantém direta e conveniada/parceira)
- `tipoesc` de Educação Infantil ou Ensino Fundamental. Ajuste as regex abaixo após ver a inspeção acima.

In [ ]:
RE_INCLUIR = r'CEI|CEMEI|EMEI|EMEF|EMEBS|CR\.?P|CRECHE|CCI|CIPS|CEU'
RE_EXCLUIR = r'CIEJA|MOVA|ESC\.? ?TEC|E ?TEC|CMCT|EJA|POLO|AT COM|PART'   # CEU POLO / AT COM = gestão, não escola
RE_INFANTIL = r'CEI|CEMEI|EMEI|CR\.?P|CRECHE|CCI|CIPS'

df = cad.copy()
if 'situacao' in df:
    df = df[df['situacao'].str.upper().str.contains('ATIV', na=False)]
if 'rede' in df:
    df = df[~df['rede'].str.upper().str.contains('PRIV|PART', na=False)]
t = df['tipoesc'].str.upper().fillna('')
df = df[t.str.contains(RE_INCLUIR) & ~t.str.contains(RE_EXCLUIR)].copy()

t = df['tipoesc'].str.upper()
df['etapa'] = np.where(t.str.contains(RE_INFANTIL), 'Infantil', 'Fundamental')
rede = df['rede'].str.upper().fillna('') if 'rede' in df else pd.Series('', index=df.index)
df['rede_tipo'] = np.where(rede.str.contains('CONV|PARC|IND') | t.str.contains(r'INDIR|CONV|CR\.?P'), 'Conveniada', 'Direta')
print(df.groupby(['etapa', 'rede_tipo']).size())
print(df['tipoesc'].value_counts())

In [ ]:
# coordenadas: vírgula decimal e/ou inteiros escalados (ex.: -23550520 → -23.55052)
def coord(s):
    v = pd.to_numeric(s.astype(str).str.replace(',', '.', regex=False), errors='coerce')
    while (v.abs() > 180).any():
        v = np.where(v.abs() > 180, v / 10, v); v = pd.Series(v, index=s.index)
    return v

df['lat'] = coord(df['latitude']); df['lon'] = coord(df['longitude'])
ok = df['lat'].between(-24.1, -23.3) & df['lon'].between(-46.9, -46.3)
print('sem coordenada válida:', (~ok).sum())
df[~ok].to_csv(f'{INT}/escolas_sem_coordenada.csv', index=False)   # geocodificar à parte se forem muitas
df = df[ok]

df = df.drop_duplicates('codesc')
esc = gpd.GeoDataFrame(df, geometry=gpd.points_from_xy(df['lon'], df['lat']), crs='EPSG:4326').to_crs(CRS)
esc = esc.rename(columns={'codesc': 'cod_esc', 'nomesc': 'nome', 'nomes': 'nome', 'codinep': 'cod_inep'})
keep = [c for c in ['cod_esc', 'cod_inep', 'nome', 'tipoesc', 'etapa', 'rede_tipo', 'dre', 'distrito', 'subpref',
                    'endereco', 'numero', 'bairro', 'lat', 'lon', 'geometry'] if c in esc.columns]
esc = esc[keep]
print(len(esc), 'escolas')

In [ ]:
# conferência com o GeoSampa (opcional): % de escolas da SME com ponto do GeoSampa a ≤ 100 m
gp = glob.glob(f'{RAW}/geosampa/*.gpkg')
if gp:
    import pyogrio
    camadas = []
    for f in gp:
        for lyr in pyogrio.list_layers(f)[:, 0]:
            camadas.append(gpd.read_file(f, layer=lyr).to_crs(CRS)[['geometry']])
    geo = pd.concat(camadas, ignore_index=True)
    j = gpd.sjoin_nearest(esc[['cod_esc', 'geometry']], geo, how='left', max_distance=100, distance_col='dist_geosampa')
    esc = esc.merge(j.groupby('cod_esc')['dist_geosampa'].min(), on='cod_esc', how='left')
    print('com correspondente no GeoSampa (≤100 m):', esc['dist_geosampa'].notna().mean().round(3))
else:
    print('Coloque os GPKG do GeoSampa em data/raw/geosampa/ para conferir.')

In [ ]:
out = f'{INT}/escolas.gpkg'
if os.path.exists(out): os.remove(out)
esc.to_file(out, layer='escolas', driver='GPKG')
for b in BUFFERS:
    buf = esc[['cod_esc', 'geometry']].copy(); buf['geometry'] = buf.buffer(b)
    buf.to_file(out, layer=f'buffer_{b}m', driver='GPKG')
esc.drop(columns='geometry').to_csv(f'{INT}/escolas.csv', index=False)
esc.plot(column='etapa', markersize=2, legend=True, figsize=(7, 9));